# NeuralEnsemble: independent fitted-model ensembles

`NeuralEnsemble` clones and independently fits any NAMpy neural regressor or
classifier, optionally on bootstrap samples. It is distinct from the jointly
trained `EnsembleTreeNAM` architecture.


## Model

$$
\widehat y(x)=\frac1M\sum_{m=1}^{M}\widehat y_m(x),
\qquad
s_t(x)=\operatorname{sd}_m\{f_{m,t}(x)\}.
$$

The first quantity is the ensemble prediction; $s_t$ measures between-member
variation of an additive term, not calibrated posterior uncertainty.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

rng = np.random.default_rng(7)
n = 180
X = pd.DataFrame({
    "x1": rng.uniform(-1.0, 1.0, n),
    "x2": rng.normal(size=n),
    "group": rng.choice(["a", "b", "c"], size=n),
})
y = (
    np.sin(np.pi * X["x1"])
    + 0.35 * X["x2"] ** 2
    + 0.30 * (X["group"] == "b")
    + rng.normal(0.0, 0.12, n)
)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=7
)

# Set True to run the small fit and all fitted-model demonstrations.
RUN_TRAINING = False


In [ ]:
from nampy.models import NAMRegressor, NeuralEnsemble

base = NAMRegressor(layer_sizes=[24, 12], dropout=0.0)
model = NeuralEnsemble(
    base,
    n_estimators=3,
    random_state=7,
    n_jobs=1,
    bootstrap=True,
)
model.get_params(deep=False)


## Fit, predict, and inspect uncertainty

Fit parameters after `y` are forwarded to every cloned member. Each member owns
its preprocessing and fitted architecture.


In [ ]:
if RUN_TRAINING:
    model.fit(
        X_train, y_train,
        max_epochs=3,
        batch_size=64,
        logger=False,
        enable_progress_bar=False,
        enable_model_summary=False,
    )
    predictions = model.predict(X_test)
    r2 = model.score(X_test, y_test)
    components = model.predict_components(X_test, center=True)
    uncertainty = model.predict_component_uncertainty(X_test, center=True)
    components.validate_additive_reconstruction()
    display({"R2": r2, "members": uncertainty.n_estimators})
    display({name: values.mean() for name, values in uncertainty.term_std.items()})


## Limits

The generic ensemble accepts regressors and classifiers. LSS aggregation is
family-specific and therefore rejected. Classification adds `predict_proba`.
